In [1]:
#%pip install torch
import torch
import torch.nn as nn
import math

In [2]:
class InputEmbedding(nn.Module):
    def __init__(self,d_model,vocab_size):
        super(InputEmbedding,self).__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size,d_model)
    def forward(self,x):
        return self.embedding(x) * math.sqrt(self.d_model)
    

In [5]:
d_model = 8
vocab_size = 1000
embed=InputEmbedding(d_model,vocab_size)
sentence_taken = torch.tensor([[10,25,500,30,31,85]])
output = embed(sentence_taken)
print(output.shape)

torch.Size([1, 6, 8])


In [6]:
print(output)

tensor([[[ 1.7494, -1.3894,  0.9013, -1.9104, -0.9152,  1.2531, -5.4752,
          -1.5565],
         [-0.8167,  2.5420, -2.3829, -1.3972, -2.0077,  2.7357,  0.8228,
          -0.9191],
         [ 1.4017,  0.8602, -5.8620,  2.5012,  2.3703, -3.6244, -2.3304,
          -0.7268],
         [-2.1583, -0.7669,  1.6259,  1.8350, -1.3079, -2.7117,  0.6998,
           1.0338],
         [-0.9153, -1.9527, -1.2778,  0.2108, -0.3655, -3.4772, -0.8725,
           1.8663],
         [ 1.0711, -3.0012, -2.1260, -3.0470,  0.2375,  1.6758, -1.8239,
           2.3827]]], grad_fn=<MulBackward0>)


In [7]:
class PositionalEncoding(nn.Module):
    def __init__(self,d_model,squ_len: int,dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.d_model = d_model
        self.squ_len = squ_len
        pe = torch.zeros(squ_len,d_model)
        position = torch.arange(0,squ_len,dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0,d_model,2).float() * (-math.log(10000.0) / d_model))
        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0,1)
        self.register_buffer('pe',pe)
        
    def forward(self,x):
        x = x + self.pe[:, :x.size(1), :].requires_grad_(False)
        return self.dropout(x)
        

In [8]:
d_model = 8
squ_len = 1000
pos_enc = PositionalEncoding(d_model,squ_len,dropout=0.1)
x=embed(sentence_taken)
output = pos_enc(x)
print(output.shape)
print(output)


torch.Size([1000, 6, 8])
tensor([[[ 1.9437e+00, -4.3268e-01,  1.0014e+00,  ...,  2.5035e+00,
          -6.0836e+00, -6.1829e-01],
         [-9.0743e-01,  3.9356e+00, -2.6477e+00,  ...,  4.1508e+00,
           9.1425e-01,  8.9862e-02],
         [ 1.5574e+00,  2.0669e+00, -6.5134e+00,  ..., -2.9160e+00,
          -0.0000e+00,  3.0360e-01],
         [-2.3981e+00,  0.0000e+00,  0.0000e+00,  ..., -1.9019e+00,
           7.7760e-01,  2.2598e+00],
         [-1.0171e+00, -1.0585e+00, -1.4197e+00,  ..., -2.7525e+00,
          -9.6946e-01,  0.0000e+00],
         [ 0.0000e+00, -2.2236e+00, -2.3622e+00,  ...,  2.9731e+00,
          -2.0266e+00,  0.0000e+00]],

        [[ 2.8787e+00, -9.4346e-01,  1.1123e+00,  ...,  2.5034e+00,
          -6.0825e+00, -6.1829e-01],
         [ 2.7533e-02,  0.0000e+00, -2.5368e+00,  ...,  4.1508e+00,
           9.1536e-01,  8.9861e-02],
         [ 2.4924e+00,  1.5562e+00, -6.4024e+00,  ..., -2.9161e+00,
          -0.0000e+00,  3.0360e-01],
         [-1.4631e+00, -2.51

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class MultiHeadAttentionBlock(nn.Module):
    def __init__(self, d_model: int, h: int, dropout: float):
        super().__init__()
        self.h = h
        self.d_model = d_model
        assert d_model % h == 0, "d_model must be divisible by h"
        
        self.head_dim = d_model // h
        
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(p=dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.size(-1)
        # Scaled Dot-Product Attention
        # (Batch, h, seq_q, d_k) x (Batch, h, d_k, seq_k) -> (Batch, h, seq_q, seq_k)
        attention_scores = torch.matmul(query, key.transpose(-2, -1)) / math.sqrt(d_k)
        
        if mask is not None:
            attention_scores = attention_scores.masked_fill(mask == 0, -1e9)
        
        attention_probs = F.softmax(attention_scores, dim=-1)
        
        if dropout is not None:
            attention_probs = dropout(attention_probs)
            
        # Multiply probs by value: (Batch, h, seq_q, seq_k) x (Batch, h, seq_k, head_dim)
        return torch.matmul(attention_probs, value), attention_probs

    def forward(self, q, k, v, mask):
        # Linear projections
        query = self.w_q(q)
        key = self.w_k(k)
        value = self.w_v(v)

        # Split into h heads: (Batch, Seq, d_model) -> (Batch, Seq, h, head_dim) -> (Batch, h, Seq, head_dim)
        query = query.view(query.size(0), -1, self.h, self.head_dim).transpose(1, 2)
        key = key.view(key.size(0), -1, self.h, self.head_dim).transpose(1, 2)
        value = value.view(value.size(0), -1, self.h, self.head_dim).transpose(1, 2)

        x, self_attention_probs = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)

        # Concatenate heads: (Batch, h, Seq, head_dim) -> (Batch, Seq, h, head_dim) -> (Batch, Seq, d_model)
        x = x.transpose(1, 2).contiguous().view(x.size(0), -1, self.d_model)

        return self.w_o(x)